# Modern LLM Orientation

The from-scratch GPT path gives us the core invariants: tokens, logits, losses, masking, optimization, and autoregressive generation. Modern LLM systems stack additional training stages and surrounding infrastructure on top of those invariants. This notebook is an orientation map for that stack, not an implementation guide for every stage.

## 1. Pretraining versus supervised fine-tuning

Pretraining teaches next-token prediction over a broad text distribution. Supervised fine-tuning narrows the behavior by training on prompt-response examples that encode task format and desired style. The base mechanism stays the same: causal language modeling over token sequences. What changes is the data distribution, the formatting convention, and the objective emphasis.

In [ ]:
topics = [
    "instruction tuning",
    "preference optimization",
    "retrieval augmented generation",
    "quantization",
    "evaluation",
]
topics

## 2. Instruction tuning

Instruction tuning is usually a supervised fine-tuning stage on curated prompt-response pairs. It teaches the model to follow explicit requests, maintain answer structure, and prefer assistant-style completions over raw continuation. The important boundary for this curriculum is that instruction tuning still depends on the same tokenizer, model, loss, and optimizer machinery you already built.

## 3. Preference optimization overview: RLHF and DPO

Preference optimization adds a second layer beyond supervised labels: pairwise or scalar signals about which outputs are better. RLHF routes that signal through a reward model and a reinforcement-learning update. DPO uses a direct preference objective without a separate online RL loop. Both are worth knowing conceptually, but they sit on top of the pretrained and supervised model rather than replacing the decoder architecture underneath.

In [ ]:
stage_map = {
    "pretraining": "broad next-token learning",
    "instruction tuning": "prompt-response behavior shaping",
    "preference optimization": "ranking outputs by human preference",
    "rag": "adding external retrieved context at inference time",
    "quantization": "compressing weights and caches for deployment constraints",
}
stage_map

## 4. Retrieval-augmented generation

RAG changes the input pipeline more than the decoder math. A retriever fetches external documents, a prompt builder inserts them into context, and the language model conditions on that expanded prompt. The core GPT still predicts the next token. The surrounding system now has failure modes in retrieval quality, chunking, stale documents, and prompt budgeting.

## 5. Quantization and serving constraints

Quantization, batching, KV-cache memory, latency targets, and accelerator availability matter once a model leaves the training notebook. Those constraints decide whether a model fits in memory, how much parallel traffic it can handle, and which decoding settings are practical. This curriculum keeps those topics at orientation depth here because Notebook 10 already handles quantization as its own technical deep dive.

## 6. Evaluation hazards

Evaluation can look rigorous while still being misleading. Prompt leakage, benchmark contamination, formatting sensitivity, cherry-picked examples, and weak human-review protocols can all inflate performance claims. Even simple metrics like loss or win rate need context: what data distribution was used, which failure cases matter, and whether the benchmark resembles the actual task.

## 7. Why architecture knowledge still matters

Modern tooling can hide the line-by-line implementation, but it cannot replace architectural judgment. When a model hallucinates, exceeds memory, slows down at long context, or behaves oddly after fine-tuning, the diagnosis still comes back to the same pieces you built from scratch: embeddings, attention, masking, optimization, and generation. That knowledge is what makes the abstraction layer usable rather than opaque.

## Abstraction Bridge

| From-scratch component | Modern layer it supports | Boundary to keep clear |
| --- | --- | --- |
| Pretraining loop | Base-model training pipelines | Data scope and objective are still your responsibility |
| Toy fine-tuning notebook | Instruction-tuning workflows | Format design and label quality dominate behavior |
| `TinyGPT` decoder | Preference-optimized chat model backbone | RLHF or DPO add objectives; they do not replace the decoder |
| `generate()` sampling loop | RAG and assistant inference stacks | Retrieval changes inputs, not the autoregressive core |
| Quantization helpers | Deployment-time compression choices | Memory savings can trade off with quality and latency |
| Validation loss and generation checks | Evaluation harnesses and leaderboards | Benchmarks can mislead without task-grounded review |